## the libraries for rl 

In [29]:
import math as m
import numpy as np
import random 
import stable_baselines3 as sb3
import copy
import gymnasium as gym
from gymnasium.vector import SyncVectorEnv, AutoresetMode
import gymnasium_robotics
import torch
import torch.nn as nn
import torch.nn.functional as F
import random 
from abc import ABC, abstractmethod


In [30]:
import torch
import torch.nn as nn

print(torch.__version__)

net = nn.Linear(4, 4)
#optimizer = torch.optim.SGD(net.parameters(), lr=3e-4)

print("works")

2.13.0+cu130
works


## all the tools needed fro RL 

### gae (generalized advantages estimation) calculation

In [31]:
def compute_gae(deltas, gamma=0.99, lam=0.95):
    """Compute GAE advantages from a list of TD residuals (deltas)"""
    advantages = torch.zeros(len(deltas))
    gae = torch.tensor(0.0)
    
    # Backward pass - this is the key
    for t in reversed(range(len(deltas))):
        gae = deltas[t] + gamma * lam * gae
        advantages[t] = gae
    
    return advantages


In [32]:
# Simulate adding steps one by one (as in your question)
deltas = []

print("Incremental GAE Update:\n")

for step in range(1, 7):
    # Simulate receiving a new step
    new_delta = round(np.random.uniform(-1.0, 2.0), 3)   # random for demo
    deltas.append(new_delta)
    
    print(f"After step {step} (new δ = {new_delta}):")
    
    # Recompute GAE using ALL data available so far
    advantages = compute_gae(deltas, gamma=0.99, lam=0.95)
    
    for i, adv in enumerate(advantages):
        print(f"   A[{i+1:2d}] = {adv:.3f}")
    print("-" * 50)

Incremental GAE Update:

After step 1 (new δ = 1.996):
   A[ 1] = 1.996
--------------------------------------------------
After step 2 (new δ = 1.724):
   A[ 1] = 3.617
   A[ 2] = 1.724
--------------------------------------------------
After step 3 (new δ = 1.19):
   A[ 1] = 4.670
   A[ 2] = 2.843
   A[ 3] = 1.190
--------------------------------------------------
After step 4 (new δ = -0.846):
   A[ 1] = 3.966
   A[ 2] = 2.095
   A[ 3] = 0.394
   A[ 4] = -0.846
--------------------------------------------------
After step 5 (new δ = 0.918):
   A[ 1] = 4.684
   A[ 2] = 2.859
   A[ 3] = 1.206
   A[ 4] = 0.017
   A[ 5] = 0.918
--------------------------------------------------
After step 6 (new δ = 0.294):
   A[ 1] = 4.901
   A[ 2] = 3.089
   A[ 3] = 1.451
   A[ 4] = 0.277
   A[ 5] = 1.195
   A[ 6] = 0.294
--------------------------------------------------


### net

In [33]:
# Uncomment if you want to use this simple version Brick 1 and 2 

# ── BRICK 1: reusable block (your Block, kept exactly) ──────────────────
class Block(nn.Module):
    """A reusable residual-style chunk: linear → norm → activation"""
    def __init__(self, dim, activation=nn.ReLU):
        super().__init__()
        self.fc   = nn.Linear(dim, dim)
        self.norm = nn.LayerNorm(dim)
        self.act  = activation()

    def forward(self, x):
        return self.act(self.norm(self.fc(x)))

# ── BRICK 2: shared body ─────────────────────────────────────────────────
class SharedBody(nn.Module):
    """
    Encodes the state φ(s) into a shared representation.
    Both actor and critic read from this.
    """
    def __init__(
        self,
        n_features: int,
        hidden_dim: int = 64,
        n_layers: int = 2,
        activation=nn.ReLU
    ):
        super().__init__()

        self.input_layer = nn.Linear(n_features, hidden_dim)

        layers = [
            self.input_layer,
            activation()
        ]

        for i in range(1, n_layers + 1):
            setattr(self, f"block{i}", Block(hidden_dim, activation))

        for i in range(1, n_layers + 1):
            layers.append(getattr(self, f"block{i}"))

        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)


# class SharedBody(nn.Module): # efficient version
#     def __init__(
#         self,
#         n_layers: int,
#         n_features: int,
#         hidden_dim: int = 64,
#         activation=nn.ReLU
#     ):
#         super().__init__()

#         self.network = nn.Sequential(
#             nn.Linear(n_features, hidden_dim),
#             activation(),
#             *[Block(hidden_dim, activation) for _ in range(n_layers)]
#         )

        
#vars() was useful here 

In [34]:

# ── BRICK 3: actor head ──────────────────────────────────────────────────
class ActorHead(nn.Module):
    """
    Takes shared body output → outputs a probability distribution over actions.
    π_θ(a|s) = softmax(W · h + b)
    """
    def __init__(self, hidden_dim: int, action_dim: int):
        super().__init__()
        self.head = nn.Linear(hidden_dim, action_dim)

    def forward(self, h):
        return F.softmax(self.head(h), dim=-1)   # shape: (action_dim,)

# added class for predicting and choosing the action in a continuous env
class GaussianActor(nn.Module):
    """
    Takes shared body output → outputs a Gaussian probability distribution over continuous actions.
    μ = W · h + b
    σ = exp(log_std)

    π_θ(a|s) = Normal(μ, σ)
    """
    def __init__(self, hidden_dim: int, action_dim: int):
            super().__init__()

            self.mean = nn.Linear(hidden_dim,action_dim)
            self.log_std = nn.Parameter(torch.zeros(action_dim))
    
    def forward(self, h):
        mean = self.mean(h)
        std = torch.exp(self.log_std)

        return torch.distributions.Normal(mean,std)


# ── BRICK 4: critic head ─────────────────────────────────────────────────
class CriticHead(nn.Module):
    """
    Takes shared body output → outputs a single scalar V(s).
    No activation — value can be any real number.
    """
    def __init__(self, hidden_dim: int):
        super().__init__()
        self.head = nn.Linear(hidden_dim, 1)

    def forward(self, h):
        return self.head(h).squeeze(-1)           # shape: scalar


# ── BRICK 5: full Actor-Critic network ───────────────────────────────────
class ActorCritic(nn.Module):
    """
    One network, two outputs:
        actor  → π_θ(a|s)   used for: action selection + policy loss
        critic → V_θ(s)     used for: GAE computation  + value loss
    """
    def __init__(self, 
                n_features: int,
                action_dim: int, 
                hidden_dim: int = 64,
                n_layers = 2,
                action_mode = "discrete"):
        super().__init__()
        self.action_mode = action_mode
        self.body   = SharedBody(n_features, hidden_dim,n_layers)
        self.critic = CriticHead(hidden_dim)
        if action_mode == "discrete":
            self.actor  = ActorHead(hidden_dim, action_dim)
        elif (action_mode== "continuous") or (action_mode == "Gauss"):
            self.actor  = GaussianActor(hidden_dim, action_dim)

        else: 
            raise ValueError(
                "actor_mode must be 'discrete' or 'continous'|'Gauss'"
            )

        self._init_weights()

    
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.orthogonal_(m.weight, gain=np.sqrt(2))
                nn.init.constant_(m.bias, 0.0)
        # small gain → initial policy near-uniform → more exploration
        nn.init.orthogonal_(self.critic.head.weight, gain=1.0)
        nn.init.constant_(self.critic.head.bias, 0.0)

        if self.action_mode == "discrete":

            nn.init.orthogonal_(self.actor.head.weight,gain=0.01)
            nn.init.constant_(self.actor.head.bias,0.0)
        else: # the case where gauß is needed
            nn.init.orthogonal_(self.actor.mean.weight,gain=0.01)
            nn.init.constant_(self.actor.mean.bias,0.0)


    def forward(self, x):
        h     = self.body(x)
        value = self.critic(h)   # V_θ(s)
        #probs = self.actor(h)    # π_θ(a|s)

        if self.action_mode == "discrete":

            probs = self.actor(h)    # π_θ(a|s)
            dist = torch.distributions.Categorical(probs)
        else : 
            dist = self.actor(h)    # π_θ(a|s)
        
        return dist, value

    def get_action(self, state_vec: np.ndarray):
        device=next(self.parameters()).device
        x = torch.tensor(state_vec,dtype=torch.float32, device = device)
        dist, value = self.forward(x)
        action = dist.sample()
        log_prob = dist.log_prob(action)

        if self.action_mode == "continuous":
            log_prob = log_prob.sum()

        if self.action_mode == "discrete":
            action = action.item()

        else:
            action = action.detach().cpu().numpy()

        return action, log_prob, value

# ── BRICK 6: your custom loss (kept from your code) ──────────────────────
class MyLoss(nn.Module):
    """MSE loss — used for critic: (V(s) - y_t)²"""
    def forward(self, y_pred, y_true):
        return ((y_pred - y_true) ** 2).mean()
    

### loss functions

In [35]:
class Loss(nn.Module):
    def __init__(self):
        super().__init__()
    def forward(self, *args, **kwargs):
        raise NotImplementedError


In [36]:
class Loss_Entropy(Loss):
    """Entropy encourages exploration."""
    def forward(self, dist) -> torch.Tensor:
        return dist.entropy().mean()


class Loss_Critic(Loss):
    """L_V = (V(s) - y)²"""
    def forward(
        self,
        v_pred: torch.Tensor,
        v_target: torch.Tensor
    ) -> torch.Tensor:

        return F.mse_loss(v_pred, v_target)


class Loss_Actor(Loss):
    """L_pi = -ρ·A — PPO clipped policy gradient"""

    def __init__(self, clip_epsilon=0.2):
        super().__init__()
        self.eps = clip_epsilon

    def forward(
        self,
        log_prob_new: torch.Tensor,
        log_prob_old: torch.Tensor,
        advantage: torch.Tensor
    ) -> torch.Tensor:

        # ρ = π_new / π_old
        rho = torch.exp(log_prob_new - log_prob_old)

        clipped = torch.clamp(
            rho,
            1 - self.eps,
            1 + self.eps
        )

        # PPO clipped objective
        L_clip = torch.min(
            rho * advantage,
            clipped * advantage
        )

        return -L_clip.mean()


class Loss_PPO(Loss):

    def __init__(self, cv=0.5, ce=0.01, clip_epsilon=0.2):
        super().__init__()

        self.cv = cv
        self.ce = ce

        self.L_pi = Loss_Actor(clip_epsilon)
        self.L_v = Loss_Critic()

    def forward(
        self,
        log_prob_new,
        log_prob_old,
        advantage,
        v_pred,
        v_target,
        entropy
    ):
        actor_loss = self.L_pi(
            log_prob_new,
            log_prob_old,
            advantage
        )

        critic_loss = self.L_v(
            v_pred,
            v_target
        )

        return (
            actor_loss
            + self.cv * critic_loss
            - self.ce * entropy
        )

### learning strategy (for improvement and update)

In [37]:
class LearningStrategy(ABC):
    @abstractmethod
    def update(self, policy, experience): pass
    @abstractmethod
    def snapshot(self): pass

In [ ]:
class PPO(LearningStrategy):
    def __init__(
        self,
        optimizer,
        clip_epsilon: float = 0.2,
        cv: float = 0.5,
        ce: float = 0.01,
        gamma: float = 0.99,
        lam: float = 0.95,
        n_epochs: int = 1,
        minibatch_size = 256
    ):
        self.optimizer = optimizer
        self.clip_epsilon = clip_epsilon
        self.cv = cv
        self.ce = ce
        self.gamma = gamma
        self.lam = lam
        self.n_epochs = n_epochs
        self.loss_fn = Loss_PPO(cv=cv, ce=ce, clip_epsilon=clip_epsilon)
        self.minibatch_size = minibatch_size

    def compute_gae(self, deltas, dones, gamma=None, lam=None):
        gamma = gamma if gamma is not None else self.gamma
        lam = lam if lam is not None else self.lam
        T = len(deltas)

        device = deltas[0].device

        advantages = torch.zeros(T,device = device)
        gae = torch.tensor(0.0,device = device)

        for t in reversed(range(T)):
            if dones[t]:
                gae = torch.tensor(0.0,device = device)
            gae = deltas[t] + gamma * lam * gae
            advantages[t] = gae.detach()
        return advantages

    def update(self, policy, rollout):
        if policy.net.action_mode == "discrete":
            return self._update_discrete(policy, rollout)
        elif policy.net.action_mode == "continuous":
            return self._update_continuous(policy, rollout)
        else:
            raise ValueError(f"Unknown action mode: {policy.net.action_mode}")

    def _update_discrete(self, policy, rollout):
        return self._update_common(policy, rollout, continuous=False)

    def _update_continuous(self, policy, rollout):
        return self._update_common(policy, rollout, continuous=True)

    def _update_common(self, policy, rollout, continuous: bool):

        device = next(policy.net.parameters()).device

        deltas = rollout["deltas"]
        states_visited = rollout["states_visited"]
        log_probs_old = rollout["log_probs_old"]
        actions_taken = rollout["actions_taken"]
        values_visited = rollout["values_visited"]
        dones = rollout["dones"]

        # ----- Advantages & Returns -----
        A = self.compute_gae(deltas, dones)
        v_tensor = torch.stack([v.detach().to(device) for v in values_visited])
        returns = (A + v_tensor).detach()

        # Normalise advantages
        if torch.isfinite(A.std()) and A.std() > 1e-8:
            A = (A - A.mean()) / (A.std() + 1e-8)
        else:
            A = A - A.mean()

        total_loss = 0.0

        for _ in range(self.n_epochs):
            log_probs_new = []
            values_new = []
            entropies = []

            for s, a in zip(states_visited, actions_taken):
                #MODIFIED
                #x = torch.as_tensor(policy.state_to_vector_fn(s),dtype=torch.float32,device=device)
                x = torch.as_tensor(s,dtype=torch.float32,device=device)
                dist, value = policy.net(x)

                if continuous:
                    action = torch.as_tensor(a, dtype=torch.float32,device=device)
                    log_prob = dist.log_prob(action).sum()
                    entropy = dist.entropy().sum()
                else:
                    #MODIFIED
                    #action_idx = policy.actions.index(a)
                    action_idx = torch.as_tensor(
                        a,
                        dtype=torch.long,
                        device=device
                        )
                    log_prob = dist.log_prob(torch.tensor(action_idx,device = device))
                    entropy = dist.entropy()

                log_probs_new.append(log_prob)
                values_new.append(value)
                entropies.append(entropy)

            log_probs_new = torch.stack(log_probs_new)
            log_probs_old_t = torch.stack([ 
                lp.to(device) for lp in log_probs_old
                ])
            v_pred = torch.stack(values_new)
            entropy = torch.stack(entropies).mean()

            loss = self.loss_fn(
                log_probs_new,
                log_probs_old_t,
                A,
                v_pred,
                returns,
                entropy
            )

            self.optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(policy.net.parameters(), max_norm=0.5)
            self.optimizer.step()

            total_loss += loss.item()

        return total_loss / self.n_epochs

    def _update_common(self, policy, rollout, continuous: bool):
        device = next(policy.net.parameters()).device

        # ----- stack everything into batch tensors -----
        states  = torch.as_tensor(np.stack(rollout["states_visited"]), dtype=torch.float32, device=device)
        actions = torch.stack([torch.as_tensor(a) for a in rollout["actions_taken"]]).to(device)
        actions = actions.float() if continuous else actions.long()
        lp_old  = torch.stack([lp.to(device) for lp in rollout["log_probs_old"]]).detach()
        values  = torch.stack([v.to(device) for v in rollout["values_visited"]]).detach()

        # ----- advantages & returns -----
        A = self.compute_gae(rollout["deltas"], rollout["dones"]).to(device)
        returns = (A + values).detach()
        A = (A - A.mean()) / (A.std() + 1e-8)

        B = states.shape[0]
        mb = getattr(self, "minibatch_size", 256)
        total_loss, n_updates = 0.0, 0

        for _ in range(self.n_epochs):
            perm = torch.randperm(B, device=device)
            for start in range(0, B, mb):
                idx = perm[start:start + mb]

                dist, value = policy.net(states[idx])
                value = value.squeeze(-1)                         # (b,)

                if continuous:
                    log_prob = dist.log_prob(actions[idx]).sum(-1)   # (b,)  same convention as rollout
                    entropy  = dist.entropy().sum(-1).mean()
                else:
                    log_prob = dist.log_prob(actions[idx])           # (b,)
                    entropy  = dist.entropy().mean()

                loss = self.loss_fn(log_prob, lp_old[idx], A[idx], value, returns[idx], entropy)

                self.optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(policy.net.parameters(), max_norm=0.5)
                self.optimizer.step()

                total_loss += loss.item()
                n_updates += 1

        return total_loss / n_updates
    def snapshot(self):
        return {
            "gamma": self.gamma,
            "lam": self.lam,
            "clip_epsilon": self.clip_epsilon,
            "cv": self.cv,
            "ce": self.ce,
            "lr": self.optimizer.param_groups[0]["lr"],
        }

### policy

In [39]:
from abc import ABC, abstractmethod

class Policy(ABC):
    @abstractmethod
    def select_action(self, state): pass
    @abstractmethod
    def action_distribution(self, state): pass
    @abstractmethod
    def snapshot(self): pass

In [40]:
def s2v(state):
    return state

class Policy_PPO(Policy):

    def __init__(self, network, actions=None,
                 state_to_vector_fn= lambda x: x):

        super().__init__()

        self.pi = {}
        self.net = network
        self.actions = actions
        self.state_to_vector_fn = state_to_vector_fn

    def select_action(self, state):

        x = self.state_to_vector_fn(state)

        action, log_prob, value = self.net.get_action(x)

        # Only discrete actions need index → environment action conversion
        if self.net.action_mode == "discrete":
            action = self.actions[action]

        return action, log_prob, value

    def action_distribution(self, state):

        x = torch.tensor(
            self.state_to_vector_fn(state),
            dtype=torch.float32
        )

        with torch.no_grad():
            dist, _ = self.net(x)

        return dist

    def snapshot(self):
        return self.net.state_dict()

### agent

In [41]:
# ════════════════════════════════════════════════════════════════════
#  BRICK 4 — AGENT
#  The self. Owns the policy (permanent). Borrows the strategy (swappable).
#  Contains no learning logic — purely coordinates the other bricks.
# ════════════════════════════════════════════════════════════════════
    
class Agent:

    def __init__(self, strategy: LearningStrategy, policy: Policy):
        self.policy   = policy    # permanent — never replaced
        self.strategy = strategy  # swappable — plug any algorithm in
        self.memory   = []        # full experience trace

    def act(self, state) -> str:
        """Ask the strategy what to do, passing the policy as context."""
        return self.policy.select_action(state)

    def learn(self, experience:dict): # experience = {"state": "A","action": "right","reward": 1,"next_state": "B","done": False}
        """Tell the strategy what happened; it writes into the policy."""
        

        self.strategy.update(self.policy, experience)
        self.memory.append((
            experience["state"].name, 
            experience["action"], 
            experience["reward"], 
            experience["next_state"].name, 
            experience["done"]
        ))

    def swap_strategy(self, new_strategy: LearningStrategy):
        """
        Replace the learning algorithm.
        The policy — and everything it has learned — is untouched.
        """
        self.strategy = new_strategy

In [42]:

class Agent_PPO(Agent):

    def __init__(self, strategy: PPO, policy: Policy_PPO):
        super().__init__(strategy, policy)

    def act(self, state):
        return self.policy.select_action(state)

    def learn(self, rollout):
        loss = self.strategy.update(self.policy, rollout)
        self.memory.append({
            "rollout_size": len(rollout["deltas"]), 
            "loss":loss
        })

    def value(self, state,device="cpu"):
        x = torch.as_tensor(
            self.policy.state_to_vector_fn(state),
            dtype=torch.float32,device=device
        )

        _, value = self.policy.net(x)
        return value

In [43]:
import gymnasium as gym
import numpy as np
import torch

def training_GymEnv(env, agent, total_steps=1_000_000, rollout_length=128,
                    options=None, device="cpu"):
    """ 
    Train a PPO agent on one or multiple Gymnasium environments. 
    
    Main idea 
    --------- 
    The function repeatedly performs two phases:
      1. Collect a rollout: 
        observation -> policy -> action -> environment -> reward + next observation
        
      2. Learn from the rollout:
        collected experience -> PPO update 
        
    Parameters 
    ---------- 
    env:
      A Gymnasium environment, a list/tuple of environments,
      or an already-created Gymnasium VectorEnv. 
    
    agent:
      The RL agent containing: - policy - neural network - learning strategy - PPO learning method 
    
    total_steps:
      Total number of environment transitions to collect. 
    
    rollout_length:
      Number of transitions collected before one PPO update. 
    
    options:
      Optional arguments passed to env.reset(). 
    
    device:
      Device used by PyTorch, e.g. "cpu" or "cuda". 
    
    """

    # ============================================================
    # 1. CREATE / USE A VECTOR ENVIRONMENT
    # ============================================================
    

    # ---------- make vec env (SAME_STEP autoreset) ----------
    if isinstance(env, gym.vector.VectorEnv):
        vec_env = env    # must have been created with autoreset_mode=SAME_STEP
    else:
        # A single environment is converted into a list containing
        # one environment.
        #
        # Example:
        #
        #     env
        #      ↓
        #   [env]
        #
        # This allows the same training code to work with either
        # one environment or multiple environments.

        envs = env if isinstance(env, (list, tuple)) else [env]
        vec_env = gym.vector.SyncVectorEnv(
            [lambda e=e: e for e in envs],
            autoreset_mode=gym.vector.AutoresetMode.SAME_STEP,
        )
    # ============================================================
    # 2. BASIC TRAINING INFORMATION
    # ============================================================

    # Number of parallel environments.
    #
    # With one environment:
    #
    #     N = 1
    #
    # With eight environments:
    #
    #     N = 8
    N = vec_env.num_envs

    # PPO discount factor.
    #
    # gamma controls how strongly future rewards are considered.    
    gamma = agent.strategy.gamma

    # Neural network used by the policy.
    #
    # Usually this contains:
    #
    #     Actor  -> action distribution
    #     Critic -> V(s)
    net = agent.policy.net

    # ============================================================
    # 3. DETERMINE ACTION SPACE
    # ============================================================

    # Action space of one environment.
    #
    # Example:
    #
    #     CartPole:
    #         Discrete(2)
    #
    #         0 = left
    #         1 = right
    space = vec_env.single_action_space

    # Determine whether the action space is discrete.
    discrete = isinstance(space, gym.spaces.Discrete)

    # Continuous environments have lower and upper action limits.
    #
    # Example:
    #
    #     Box([-1, -1], [1, 1])    
    if not discrete:
        low, high = space.low, space.high
  

    # ============================================================
    # 4. RESET ENVIRONMENT
    # ============================================================

    # Obtain the initial observation.
    #
    # With one environment:
    #
    #     obs.shape = (1, observation_dimension)
    #
    # Example for CartPole:
    #
    #     obs =
    #     [[cart_position,
    #       cart_velocity,
    #       pole_angle,
    #       pole_angular_velocity]]
    obs, _ = vec_env.reset(options=options)

    # Number of transitions/PPO Updates collected so far .
    steps_done, update_count = 0, 0

    # ============================================================
    # 5. MAIN TRAINING LOOP
    # ============================================================
    while steps_done < total_steps:
      
        # --------------------------------------------------------
        # Rollout buffers
        # --------------------------------------------------------
        #
        # These store everything needed for the PPO update.
        #
        # S  = states
        # A  = actions
        # R  = rewards
        # LP = old log probabilities
        # V  = old value predictions
        # D  = TD errors / deltas
        # DN = done flags
        #
        S, A, R, LP, V, D, DN = [], [], [], [], [], [], []
        # S: states , A: actions , R: reward ,LP: log_prob, V: values , D:deltas, DN:dones 

        # ========================================================
        # 6. COLLECT ROLLOUT
        # ========================================================
        for _ in range(rollout_length):

            # ----------------------------------------------------
            # Convert observation to PyTorch tensor
            # ----------------------------------------------------
            x = torch.as_tensor(obs, dtype=torch.float32, device=device)
            
            # ----------------------------------------------------
            # POLICY + CRITIC
            # ----------------------------------------------------

            # We are collecting experience, not training yet.
            #
            # Therefore gradients are not required here.

            with torch.no_grad():
                
                # The network produces:
                #
                #     dist  = action distribution
                #     value = V(s)
                dist, value = net(x)

                # Convert:
                #
                #     (N, 1)
                #
                # into:
                #
                #     (N,)
                #
                value = value.squeeze(-1)                  # (N,)
                # ------------------------------------------------
                # SAMPLE ACTION
                # ------------------------------------------------
                #
                # Discrete:
                #
                #     action ~ Categorical
                #
                # Continuous:
                #
                #     action ~ Normal
                action = dist.sample()

                # ------------------------------------------------
                # LOG PROBABILITY
                # ------------------------------------------------
                #
                # Store the probability of the selected action
                # under the CURRENT/OLD policy.
                #
                # PPO will later compare this against the new
                # policy probability.
                log_prob = dist.log_prob(action)
                # For multidimensional continuous actions:
                #
                #     [linear_velocity,
                #      angular_velocity]
                #
                # we need one log probability for the complete
                # action.
                if log_prob.dim() > 1:
                    log_prob = log_prob.sum(-1)            # (N,)

            # ====================================================
            # 7. CONVERT ACTION TO NUMPY
            # ====================================================
            act = action.cpu().numpy()
            
            # ----------------------------------------------------
            # Continuous action clipping
            # ----------------------------------------------------
            #
            # Make sure the action is inside the Gymnasium
            # action space limits.
            if not discrete:
                act = np.clip(act.reshape(N, *space.shape), low, high)

            # ====================================================
            # 8. STEP ENVIRONMENT
            # ====================================================

            # The environment receives the action and returns:
            #
            #     next observation
            #     reward
            #     terminated
            #     truncated
            #     info
            next_obs, reward, terminated, truncated, info = vec_env.step(act)
            
            # Episode is finished if either:
            #
            #     terminated = True
            #
            # or
            #
            #     truncated = True
            done = terminated | truncated

            # ====================================================
            # 9. GET REAL FINAL OBSERVATION
            # ====================================================

            # SAME_STEP autoreset means that a finished environment
            # may already contain the FIRST observation of the next
            # episode in next_obs.
            #
            # However, we sometimes need the actual final state
            # of the previous episode.
            #
            # That state is stored in:
            #
            #     info["final_obs"]
            real_next = np.array(next_obs, copy=True)

            # ====================================================
            # 10. CALCULATE NEXT STATE VALUE
            # ====================================================
            for i in np.where(done)[0]:
                real_next[i] = info["final_obs"][i]

            with torch.no_grad():

                # Critic estimates:
                #
                #     V(next_state)
                _, nv = net(torch.as_tensor(real_next, dtype=torch.float32, device=device))
                nv = nv.squeeze(-1)

                # ------------------------------------------------
                # TERMINATED vs TRUNCATED
                # ------------------------------------------------
                #
                # True termination:
                #
                #     no future value
                #
                #     V(next_state) = 0
                #
                # Truncation:
                #
                #     episode was stopped artificially
                #
                #     future value can still matter
                #
                # Therefore we only set the value to zero for
                # genuinely terminated environments.
                nv = torch.where(torch.as_tensor(terminated, device=device),
                                 torch.zeros_like(nv), nv)   # timeout still bootstraps

            # Convert reward to PyTorch tensor
            r = torch.as_tensor(reward, dtype=torch.float32, device=device)

            # ====================================================
            # 11. STORE EXPERIENCE
            # ====================================================

            # Current state       #Action selected by the policy  #Reward from the env
            S.append(obs.copy());  A.append(action);               R.append(r)

            #Old policy log probability  #Critic's prediction V(s) 
            LP.append(log_prob);          V.append(value);         
            
            # ----------------------------------------------------
            # TD ERROR / DELTA
            # ----------------------------------------------------
            #
            #     delta =
            #
            #         reward
            #         + gamma * V(next_state)
            #         - V(current_state)
            #
            # Interpretation:
            #
            #     positive delta
            #         -> outcome was better than expected
            #
            #     negative delta
            #         -> outcome was worse than expected
            D.append(r + gamma * nv - value)

            # Whether the environment episode ended.
            DN.append(done)

            # Move to the next state
            obs = next_obs

        # ========================================================
        # 12. UPDATE STEP COUNT
        # ========================================================

        # rollout_length transitions were collected from EACH
        # environment.
        #
        # Example:
        #
        #     N = 8
        #     rollout_length = 128
        #
        #     8 * 128 = 1024 transitions
        steps_done += N * rollout_length

        # ========================================================
        # 13. PREPARE ROLLOUT DATA
        # ========================================================

        # (T, N, ...) -> env-major (N*T, ...), so each env's steps are contiguous
        T = rollout_length

        # The rollout is originally stored as:
        #
        #     (T, N, ...)
        #
        # We transform it into:
        #
        #     (N*T, ...)
        #
        # while keeping each environment's trajectory together.
        flat = lambda xs: torch.stack(xs).transpose(0, 1).reshape(N * T, *xs[0].shape[1:])

        # ========================================================
        # 14. PREPARE DONE FLAGS
        # ========================================================

        # Original:
        #
        #     (T, N)
        #
        # becomes:
        #
        #     (N, T)
        #
        # so that each environment's trajectory is contiguous.
        dones = np.stack(DN).T.copy()        # (N, T)

        # --------------------------------------------------------
        # End of rollout = GAE boundary
        # --------------------------------------------------------
        #
        # Even if the actual environment episode has NOT ended,
        # we stop the GAE calculation at the end of this rollout.
        #
        # This prevents the advantage calculation from crossing
        # into the next rollout.
        dones[:, -1] = True                  # cut GAE at the end of each env's segment

        # ========================================================
        # 15. SEND EXPERIENCE TO PPO
        # ========================================================
        agent.learn({
            # States:
            #
            #     (N*T, observation_dimension)
            #  
            "states_visited": list(np.stack(S).transpose(1, 0, 2).reshape(N * T, -1)),
            # Actions selected by the old policy.
            "actions_taken":  list(flat(A)),

            #Rewards received.
            "rewards":        list(flat(R)),

            # Log probabilities under the OLD policy.
            #
            # PPO uses these to calculate the probability ratio:
            #
            #     new probability
            #     ----------------
            #     old probability
            #
            "log_probs_old":  list(flat(LP)),

            #Critic's original value estimates.
            "values_visited": list(flat(V)),

            # TD errors / deltas.
            #
            # These are later used to calculate GAE advantages.
            "deltas":         list(flat(D)),

            # Episode / rollout boundaries.
            "dones":          dones.reshape(-1).tolist(),
        })

        # One PPO learning update is complete.
        update_count += 1

        # ========================================================
        # 16. TRAINING PROGRESS
        # ========================================================
        
        if update_count % 20 == 0:
            print(f"[steps {steps_done}/{total_steps}] [updates {update_count}]")

    # ============================================================
    # 17. TRAINING FINISHED
    # ============================================================
    print(f"Training done. {steps_done} transitions, {update_count} PPO updates.")

    # ============================================================
    # 18. CLOSE VECTOR ENVIRONMENT
    # ============================================================

    # If this function created the VectorEnv, close it here.
    #
    # If the user supplied an existing VectorEnv, the caller owns
    # it, so we do not close it here.
    if not isinstance(env, gym.vector.VectorEnv):
        vec_env.close()

## here is the env from mujoco

In [44]:
#env = gym.make("CartPole-v1")
env = gym.make("InvertedPendulum-v5")#,render_mode = "human") # better not enable the human mode for rendering , it crashes te kernel 
print("observation space",env.observation_space.shape[0])
print("action space",env.action_space.shape[0])
print("reward",)

observation space 4
action space 1
reward


In [45]:
# Here as a dashboard for developping the customable reward

import inspect
import gymnasium as gym
from gymnasium import spaces
import numpy as np

### custom obs
class ObservationCustom(gym.ObservationWrapper):

    def __init__(self, env, s2v):

        super().__init__(env)

        self.s2v = s2v

        # Determine size of the new observation
        example = self.s2v(env.observation_space.sample())

        self.observation_space = spaces.Box(
            low=-np.inf,
            high=np.inf,
            shape=example.shape,
            dtype=np.float32
        )

    def observation(self, obs):

        return self.s2v(obs).astype(np.float32)

### custom reward

class RewardCustom(gym.Wrapper):
    """
    reward_fn can take any subset of these argument names (or **kwargs for all):
    obs, action, next_obs, reward, terminated, truncated, info, env
    """
    def __init__(self, env, reward_fn):
        super().__init__(env)
        self.reward_fn = reward_fn
        params = inspect.signature(reward_fn).parameters
        self._takes_all = any(p.kind == p.VAR_KEYWORD for p in params.values())
        self._params = set(params)
        self._obs = None

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        self._obs = obs
        return obs, info

    def step(self, action):
        u = self.env.unwrapped
        state = getattr(u, "state",None) # None for envs without .state

        next_obs, reward, terminated, truncated, info = self.env.step(action)

        next_state = getattr(u, "state",None)

        ctx = dict(
            state=state,
            action=action,
            next_state=next_state,
            obs=self._obs,
            next_obs=next_obs,
            reward=reward,
            terminated=terminated,
            truncated=truncated,
            info=info,
            env=u,
        )

        kwargs = (
            ctx if self._takes_all
            else {k: v for k, v in ctx.items() if k in self._params}
        )

        info["original_reward"] = reward

        new_reward = float(self.reward_fn(**kwargs))

        self._obs = next_obs

        return next_obs, new_reward, terminated, truncated, info

In [46]:
# Here is to design the custom reward / obs functiond

## obs func

def s2v_fn(obs,*args,**kwargs):
    return obs # modifiable function 

## reward func

def reward_func(reward,*args,**kwargs):
    return reward # modifiable function 

In [47]:
def make_vec_env(base_env, n, **extra):

    template = copy.deepcopy(base_env) # pristine snapshot; fails here if not copyable

    def clone(): return copy.deepcopy(template) # each sub-env gets its own copy

    return SyncVectorEnv([clone]*n,autoreset_mode=AutoresetMode.SAME_STEP, **extra)

# # Usage

# base_env = gym.make("CartPole-v1")
# base_env = RewardCustom(base_env, my_reward)     # wrappers applied beforehand
# base_env = ObservationCustom(base_env, my_s2v)

# vec = make_vec_env(base_env, 8)
# obs, _ = vec.reset(seed=0)                       # always seed: env i gets seed 0+i
# print(vec.single_observation_space, vec.single_action_space)
# print(vec.num_envs)

In [48]:

#env = gym.make("CartPole-v1")
inv_env = gym.make("InvertedPendulum-v5",)#render_mode = "human")
#print(env.action_space)

### space to plug in the reward function and observation function but how to ?

# env = ObservationCustom(env, s2v_fn=my_s2v) # the function s2v is designed on a specific cell 
# env = RewardCustom(env,reward_fn = my_reward) # the function reward is designed on a specific cell

### Part to generate an env_vec from the base_env

N_ENVS = 8

vec_env = make_vec_env(inv_env,n= N_ENVS)

#### CartPole (discrete)
# vec = gym.make_vec("CartPole-v1", num_envs=8, vectorization_mode="sync",
#                   vector_kwargs={"autoreset_mode": gym.vector.AutoresetMode.SAME_STEP})

#### InvertedPendulum (continuous)
#vec = gym.make_vec("InvertedPendulum-v5", num_envs=8, vectorization_mode="sync",
#                   vector_kwargs={"autoreset_mode": gym.vector.AutoresetMode.SAME_STEP})

# sanity check: copies are independent
obs, _ = vec_env.reset(seed=0)
assert len({id(e.unwrapped) for e in vec_env.envs}) == N_ENVS, "sub-envs are not distinct"
assert not np.allclose(obs[0], obs[1]), "sub-envs share the same RNG"
print(obs.shape, vec_env.single_observation_space, vec_env.single_action_space)

# ========================================================= 
#  PPO AGENT 
# ========================================================= 

# Choose device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# device = torch.device("cpu")   # force CPU
# device = torch.device("cuda")  # force CUDA



print("Using ", device)

n_features = inv_env.observation_space.shape[0] #env.observation_space.shape[0] 
action_dim = inv_env.action_space.shape[0] #env.action_space.shape[0]
net = ActorCritic( n_features=n_features, 
                  action_dim=action_dim, 
                  hidden_dim=64 ,
                  action_mode="continuous"
                  ).to(device)

policy = Policy_PPO(net,state_to_vector_fn= lambda x:x ) 
optimizer = torch.optim.Adam( net.parameters(), lr=3e-4 ) 
ppo = PPO( optimizer=optimizer, clip_epsilon=0.2, gamma=0.9, lam=0.95, cv=0.5, ce=0.01, n_epochs=10 ) 
agent_gym = Agent_PPO( strategy=ppo, policy=policy )

(8, 4) Box(-inf, inf, (4,), float64) Box(-3.0, 3.0, (1,), float32)
Using  cpu


In [50]:
# =========================================================
#  TRAIN  (tiny test first, then the real run)
# =========================================================
# training_GymEnv(vec_env, agent_gym, total_steps=N_ENVS * 256 * 2, rollout_length=256, device=device)

training_GymEnv(
    vec_env, agent_gym,
    total_steps=500_000,
    rollout_length=256,        # steps PER ENV with the version I sent -> 8 * 256 = 2048 samples/update
    device=device,
)

vec_env.close()

ValueError: The value argument to log_prob must be a Tensor

In [49]:
N = 10
steps = 10
for episode in range(1,N+1):
    state, info = inv_env.reset()

    print(f"info start from reset of episode {episode} : {info}")

    done = False
    score = 0

    while not done:
        action, log_prob, value = agent_gym.act(state)

        obs, reward, terminated, truncated, info = inv_env.step(action)

        state = obs
        done = terminated or truncated

        score += reward

        print(
            f"action: {action}\n"
            f"obs: {obs}\n"
            f"reward: {reward}\n"
            f"terminated?: {terminated}\n"
            f"truncated?: {truncated}\n"
            f"info: {info}\n"
        )
    inv_env.close()
    print(f"Total Score: {score}")

info start from reset of episode 1 : {}
action: [-0.03878154]
obs: [ 0.0083478   0.00176163 -0.01460488  0.03061119]
reward: 1
terminated?: False
truncated?: False
info: {'reward_survive': 1}

action: [0.5478539]
obs: [ 0.01140489 -0.00539232  0.1669537  -0.38390781]
reward: 1
terminated?: False
truncated?: False
info: {'reward_survive': 1}

action: [-0.8755186]
obs: [ 0.01220916 -0.00687141 -0.12565189  0.3000537 ]
reward: 1
terminated?: False
truncated?: False
info: {'reward_survive': 1}

action: [0.03790326]
obs: [ 0.00749235  0.0040267  -0.11047598  0.24774576]
reward: 1
terminated?: False
truncated?: False
info: {'reward_survive': 1}

action: [-0.0154016]
obs: [ 0.00299588  0.01397469 -0.11453248  0.25157677]
reward: 1
terminated?: False
truncated?: False
info: {'reward_survive': 1}

action: [1.2908685]
obs: [ 0.00700405  0.00423664  0.31361778 -0.7267117 ]
reward: 1
terminated?: False
truncated?: False
info: {'reward_survive': 1}

action: [0.29382992]
obs: [ 0.02137916 -0.0282370